# U.S. Patent Phrase to Phrase Matching
## 02 - Baseline Model (Sentence-BERT)

Quick baseline using Sentence-BERT embeddings and cosine similarity.

In [ ]:
# Install sentence-transformers if needed
!pip install sentence-transformers -q

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('../..')
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output'

## Load Data

In [ ]:
# Load data
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {len(train_df)} rows')
print(f'Test: {len(test_df)} rows')

## Load Sentence-BERT Model

In [ ]:
# Load model - using MiniLM (fast)
print('Loading Sentence-BERT model...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded!')

## Encode Training Data

In [ ]:
# Encode anchor and target separately
print('Encoding training anchors...')
train_anchor_emb = model.encode(
    train_df['anchor'].tolist(),
    show_progress_bar=True
)

print('Encoding training targets...')
train_target_emb = model.encode(
    train_df['target'].tolist(),
    show_progress_bar=True
)

## Compute Training Similarity Scores

In [ ]:
# Compute cosine similarity for training
train_scores = []
for i in range(len(train_df)):
    anchor_emb = train_anchor_emb[i:i+1]
    target_emb = train_target_emb[i:i+1]
    sim = cosine_similarity(anchor_emb, target_emb)[0, 0]
    train_scores.append(sim)

train_df['pred_score'] = train_scores

# Calculate Pearson correlation
corr, pval = pearsonr(train_df['score'], train_df['pred_score'])
print(f'Training Pearson Correlation: {corr:.4f}')

## Encode Test Data and Generate Predictions

In [ ]:
# Encode test data
print('Encoding test anchors...')
test_anchor_emb = model.encode(
    test_df['anchor'].tolist(),
    show_progress_bar=True
)

print('Encoding test targets...')
test_target_emb = model.encode(
    test_df['target'].tolist(),
    show_progress_bar=True
)

In [ ]:
# Compute test predictions
test_scores = []
for i in range(len(test_df)):
    anchor_emb = test_anchor_emb[i:i+1]
    target_emb = test_target_emb[i:i+1]
    sim = cosine_similarity(anchor_emb, target_emb)[0, 0]
    test_scores.append(sim)

# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'score': test_scores
})

print(f'Submission shape: {submission.shape}')
print(submission.head())

In [ ]:
# Save baseline submission
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_DIR / 'submission_baseline.csv', index=False)
print(f'Baseline saved to {OUTPUT_DIR / "submission_baseline.csv"}')

## Results Summary

In [ ]:
print('=' * 50)
print('BASELINE RESULTS')
print('=' * 50)
print(f'Model: all-MiniLM-L6-v2')
print(f'Training Pearson: {corr:.4f}')
print(f'Expected LB: ~0.60')
print('=' * 50)